In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import os

# ===============================
# Parameters
# ===============================

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50

dataset_path = "/kaggle/input/datasets/anandjain1112/disease-56-crop-9/Multi indo 9 crops"

# ===============================
# Dataset Loading
# ===============================

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)

# ===============================
# Prefetch for performance
# ===============================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# ===============================
# Single Head Self Attention Layer
# ===============================

class SingleHeadSelfAttention(tf.keras.layers.Layer):

    def __init__(self, embed_dim):
        super().__init__()
        self.query = tf.keras.layers.Dense(embed_dim)
        self.key = tf.keras.layers.Dense(embed_dim)
        self.value = tf.keras.layers.Dense(embed_dim)

    def call(self, inputs):

        Q = self.query(inputs)
        K = self.key(inputs)
        V = self.value(inputs)

        score = tf.matmul(Q, K, transpose_b=True)
        scale = tf.math.sqrt(tf.cast(tf.shape(K)[-1], tf.float32))

        attention_weights = tf.nn.softmax(score / scale)

        output = tf.matmul(attention_weights, V)

        return output

# ===============================
# MobileNetV2 Base Model
# ===============================

base_model = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = True

# ===============================
# Model Architecture
# ===============================

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

x = base_model(x)

# CNN feature map → sequence

x = tf.keras.layers.Reshape((-1, x.shape[-1]))(x)

# Single Head Transformer Attention

x = SingleHeadSelfAttention(embed_dim=128)(x)

# Global Pooling

x = tf.keras.layers.GlobalAveragePooling1D()(x)

# Dense Layer + Dropout

x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.8)(x)

# Output Layer

outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

# ===============================
# Compile Model
# ===============================

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ===============================
# Train Model
# ===============================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ===============================
# Accuracy and Loss Graph
# ===============================

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])

plt.show()

# ===============================
# Prediction
# ===============================

y_true = []
y_pred = []

for images, labels in val_ds:

    preds = model.predict(images, verbose=0)
    preds = np.argmax(preds, axis=1)

    y_pred.extend(preds)
    y_true.extend(labels.numpy())

# ===============================
# Classification Report
# ===============================

print(classification_report(y_true, y_pred, target_names=class_names))

# ===============================
# Confusion Matrix
# ===============================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()